# AI Adoption in Algeria

In this analysis, we explore the adoption of AI in Algeria, examining how much these tools are being used, what they are being used for, and how Algeria's usage compares to other countries. The analysis draws on two sources:

- Anthropic Economic Index: a public release of Claude conversations classified by country, occupational task, and user intent. The AEI is the primary source because it offers more granular data on usage patterns.
- OpenAI Signals: country-level rankings of ChatGPT usage in 2025. It's used as a secondary source to provide a broader benchmark of AI adoption, because Claude alone is not representative of the broader AI market.

And the analysis is organized around three questions:

1. How much is Algeria using AI?
2. What is Algeria using AI for?
3. What is distinctive about Algeria's usage?

It's worth noting that the Claude conversations analysed here cover a single week (5–12 February 2026), and the OpenAI ranking is annual. Therefore, the findings are a snapshot of AI adoption at a specific point in time.

## Setup

In [1]:
from pathlib import Path

import altair as alt
import pandas as pd
import squarify
import attaviz

attaviz.enable()


In [2]:
def clean_names(df):
    """Standardize column names: lowercase, strip, replace spaces with underscores."""
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    return df


def find_project_root(marker="pyproject.toml"):
    """Walk up from this notebook's directory until we find the project root."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

ANTHROPIC_DIR = DATA_DIR / "AI" / "Anthropic"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

## Data and Functions

In [3]:
df = pd.read_csv(
    ANTHROPIC_DIR
    / "release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
)
dza = df.copy().query("geo_id == @COUNTRY_CODE")

In [4]:
FONT_SIZE = 11
CHAR_W = FONT_SIZE * 0.62
LINE_H = FONT_SIZE * 1.35
PAD = 6
SAFETY_PX = 4

# Bars whose value is at least this fraction of the x-axis max get their value
# label rendered *inside* the bar (white, right-aligned). Shorter bars get a
# dark label *outside* the bar end. Tuned to roughly match Datawrapper output.
INSIDE_LABEL_THRESHOLD = 0.20


def _wrap_label(text: str, value: float, dx: float, dy: float) -> str | None:
    """Word-wrap `text` to fit a (dx, dy) rectangle and append the % value.
    Returns None if the rectangle can't safely hold a label + value.
    """
    avail_w = dx - 2 * PAD - SAFETY_PX
    avail_h = dy - 2 * PAD
    if avail_w <= 0 or avail_h <= 0:
        return None

    max_chars = int(avail_w // CHAR_W)
    max_lines = int(avail_h // LINE_H)
    if max_chars < 3 or max_lines < 1:
        return None

    value_str = f"{value:.1f}%"
    if len(value_str) > max_chars:
        return None

    words = text.split()
    # Refuse to render if the longest single word can't fit on a line —
    # avoids visible mid-word truncation (`Computer…`) that still leaks.
    if any(len(w) > max_chars for w in words):
        return None

    lines: list[str] = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            lines.append(current)
            current = word
            if len(lines) >= max_lines - 1:
                # No more room for label lines; reserve the last line for value
                break
    if current and len(lines) < max_lines - 1:
        lines.append(current)

    if not lines:
        return None
    lines.append(value_str)
    return "\n".join(lines)


def make_treemap(
    data: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    subtitle: str | None = None,
    width: int = 720,
    height: int = 480,
    drop_not_classified: bool = True,
) -> alt.Chart:
    """Render a treemap from a (label, value) DataFrame using squarify + Altair.

    Labels are word-wrapped to fit each rectangle; rectangles too small to hold
    a label safely keep their tooltip but show no text.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[
            lambda df: (
                ~df[label_col].str.contains("not_classified", case=False, na=False)
            )
        ]
    d = (
        d.loc[lambda df: df[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )

    norm_values = squarify.normalize_sizes(d[value_col].tolist(), width, height)
    rects = squarify.squarify(norm_values, 0, 0, width, height)
    coords = pd.DataFrame(rects).assign(
        x2=lambda df: df["x"] + df["dx"], y2=lambda df: df["y"] + df["dy"]
    )
    plot_df = pd.concat([d, coords], axis=1)
    plot_df["label"] = plot_df.apply(
        lambda r: _wrap_label(str(r[label_col]), r[value_col], r["dx"], r["dy"]),
        axis=1,
    )

    base = alt.Chart(plot_df).encode(
        x=alt.X("x:Q", axis=None, scale=alt.Scale(domain=[0, width])),
        x2="x2:Q",
        y=alt.Y("y:Q", axis=None, scale=alt.Scale(domain=[0, height], reverse=True)),
        y2="y2:Q",
    )
    rects_layer = base.mark_rect(stroke="white", strokeWidth=2).encode(
        color=alt.Color(
            f"{label_col}:N",
            legend=None,
            scale=alt.Scale(range=attaviz.CATEGORICAL_TEXT),
        ),
        tooltip=[
            alt.Tooltip(f"{label_col}:N", title="Group"),
            alt.Tooltip(f"{value_col}:Q", title="% of conversations", format=".2f"),
        ],
    )
    text_layer = (
        base.transform_filter("datum.label != null")
        .mark_text(
            align="left",
            baseline="top",
            dx=PAD,
            dy=PAD,
            fontSize=FONT_SIZE,
            lineBreak="\n",
            color="white",
        )
        .encode(x="x:Q", y="y:Q", text="label:N")
    )
    if subtitle is None:
        subtitle = ""
    return (rects_layer + text_layer).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


def make_bar(
    data: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    subtitle: str | None = None,
    width: int = 640,
    drop_not_classified: bool = True,
    top_n: int | None = None,
) -> alt.Chart:
    """Label above each bar (editorial style).

    Always readable regardless of label or bar length, but the chart is taller
    because each row reserves space for label + bar.
    """
    d = data.copy()
    if drop_not_classified:
        d = d.loc[
            lambda df: (
                ~df[label_col].str.contains("not_classified", case=False, na=False)
            )
        ]
    d = (
        d.loc[lambda df: df[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None:
        d = d.head(top_n)

    row_h = 38
    bar_size = 14
    height = max(200, row_h * len(d))
    x_max = d[value_col].max() * 1.02
    inside_cutoff = x_max * INSIDE_LABEL_THRESHOLD

    plot_df = d.assign(
        _zero=0.0,
        _value_label=d[value_col].map(lambda v: f"{v:.1f}%"),
    )
    # Explicit sort list (descending by value) — robust across filtered layers.
    sort_order = plot_df[label_col].tolist()
    y_enc = alt.Y(f"{label_col}:N", sort=sort_order, title=None, axis=None)

    bars = (
        alt.Chart(plot_df)
        .mark_bar(size=bar_size)
        .encode(
            x=alt.X(
                f"{value_col}:Q",
                title="% of conversations",
                scale=alt.Scale(domain=[0, x_max], nice=False),
            ),
            y=y_enc,
            tooltip=[
                alt.Tooltip(f"{label_col}:N", title="Group"),
                alt.Tooltip(f"{value_col}:Q", title="%", format=".2f"),
            ],
        )
    )
    label_above = (
        alt.Chart(plot_df)
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=FONT_SIZE,
            color="#222",
            dy=-(bar_size // 2) - 2,
        )
        .encode(x="_zero:Q", y=y_enc, text=f"{label_col}:N")
    )
    # Long bars: white label tucked inside the right edge of the bar.
    pct_inside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] >= {inside_cutoff}")
        .mark_text(
            align="right",
            baseline="middle",
            dx=-PAD,
            color="white",
            fontSize=FONT_SIZE - 1,
            fontWeight="bold",
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    # Short bars: dark label sitting just past the bar end.
    pct_outside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] < {inside_cutoff}")
        .mark_text(
            align="left",
            baseline="middle",
            dx=PAD,
            color="#444",
            fontSize=FONT_SIZE - 1,
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    if subtitle is None:
        subtitle = ""
    return (bars + label_above + pct_inside + pct_outside).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)
    )


## How much is Algeria using AI?

The first question centered around the magnitude of AI usage in Algeria. To answer this, we looked at three different metrics:

- Raw Claude usage share: the share of all Claude conversations originating in Algeria.
- Claude Usage Index (AUI): usage share normalised by working-age population. An index of 1.0 means a country's Claude use matches its share of the world's working-age population, above 1.0 means more intensive use, below means less.
- ChatGPT country rank: Algeria's position in OpenAI's 2025 ranking of ChatGPT-using countries, converted to a percentile.

Each view is shown against three peer groups, with Algeria as the reference in every panel:

- Regional peers: Egypt, Jordan, Morocco, Tunisia, Iraq.
- Structural peers: Ecuador, Peru, Ghana, Vietnam, Colombia.
- Aspirational peers: Malaysia, Chile, Poland, Romania.

In [5]:
PEER_GROUPS = {
    "Regional": ["EG", "JO", "MA", "TN", "IQ"],
    "Structural": ["EC", "PE", "GH", "VN", "CO"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
COUNTRY_NAMES = {
    "DZ": "Algeria",
    "EG": "Egypt",
    "JO": "Jordan",
    "MA": "Morocco",
    "TN": "Tunisia",
    "IQ": "Iraq",
    "EC": "Ecuador",
    "PE": "Peru",
    "GH": "Ghana",
    "VN": "Vietnam",
    "CO": "Colombia",
    "MY": "Malaysia",
    "CL": "Chile",
    "PL": "Poland",
    "RO": "Romania",
}
GROUP_OF = {c: g for g, members in PEER_GROUPS.items() for c in members}
GROUP_OF[COUNTRY_CODE] = "Algeria"  # Algeria gets its own category for highlighting
GROUP_ORDER = ["Algeria", "Regional", "Structural", "Aspirational"]
GROUP_COLORS = {
    "Algeria": "#0071BC",
    "Regional": "#8A969F",
    "Structural": "#8A969F",
    "Aspirational": "#8A969F",
}

PEER_CODES = [COUNTRY_CODE] + [c for g in PEER_GROUPS.values() for c in g]

In [6]:
def make_peer_panel(
    data: pd.DataFrame,
    value_col: str,
    title: str,
    subtitle: str,
    value_axis_title: str,
    value_format: str = ".2f",
    width: int = 480,
    parity_at: float | None = None,
    label_col: str | None = None,
) -> alt.Chart:
    """One vconcat with three panels (Regional / Structural / Aspirational)
    each comparing Algeria against the countries in that peer group.

    Each panel sorts independently by `value_col`, so Algeria's rank within the
    group is read directly off the panel. `parity_at` draws a dashed reference
    line (e.g. AUI parity at 1.0). `label_col` overrides the end-of-bar label
    with a pre-formatted string column (used for the "1.78×" AUI labels).
    Labels follow Datawrapper convention: long bars get a white label tucked
    inside the bar end; short bars get a dark label just past the bar end.
    """
    color_enc = alt.Color(
        "group:N",
        scale=alt.Scale(
            domain=GROUP_ORDER, range=[GROUP_COLORS[g] for g in GROUP_ORDER]
        ),
        legend=None,
    )

    panels = []
    for group_name in ["Regional", "Structural", "Aspirational"]:
        codes_in_panel = [COUNTRY_CODE] + PEER_GROUPS[group_name]
        d = (
            data.loc[data["geo_id"].isin(codes_in_panel)]
            .sort_values(value_col, ascending=False)
            .reset_index(drop=True)
        )
        # Unify label source so inside/outside layers can read the same field.
        if label_col is not None:
            d = d.assign(_label=d[label_col].astype(str))
        else:
            d = d.assign(_label=d[value_col].map(lambda v: format(v, value_format)))

        row_h = 26
        panel_height = max(120, row_h * len(d))
        panel_x_max = d[value_col].max() * 1.05
        inside_cutoff = panel_x_max * INSIDE_LABEL_THRESHOLD

        y_enc = alt.Y(
            "country_name:N",
            sort=alt.SortField(field=value_col, order="descending"),
            title=None,
            axis=alt.Axis(grid=False),
        )
        bars = (
            alt.Chart(d)
            .mark_bar(size=14)
            .encode(
                x=alt.X(
                    f"{value_col}:Q",
                    title=value_axis_title,
                    scale=alt.Scale(domain=[0, panel_x_max], nice=False),
                ),
                y=y_enc,
                color=color_enc,
                tooltip=[
                    alt.Tooltip("country_name:N", title="Country"),
                    alt.Tooltip("group:N", title="Group"),
                    alt.Tooltip(
                        f"{value_col}:Q",
                        title=value_axis_title,
                        format=value_format,
                    ),
                ],
            )
        )
        labels_inside = (
            alt.Chart(d)
            .transform_filter(f"datum['{value_col}'] >= {inside_cutoff}")
            .mark_text(
                align="right",
                baseline="middle",
                dx=-PAD,
                color="white",
                fontSize=FONT_SIZE - 1,
                fontWeight="bold",
            )
            .encode(x=f"{value_col}:Q", y=y_enc, text="_label:N")
        )
        labels_outside = (
            alt.Chart(d)
            .transform_filter(f"datum['{value_col}'] < {inside_cutoff}")
            .mark_text(
                align="left",
                baseline="middle",
                dx=PAD,
                color="#444",
                fontSize=FONT_SIZE - 1,
                fontWeight="bold",
            )
            .encode(x=f"{value_col}:Q", y=y_enc, text="_label:N")
        )

        layers = [bars, labels_inside, labels_outside]
        if parity_at is not None:
            parity = (
                alt.Chart(pd.DataFrame({"x": [parity_at]}))
                .mark_rule(color="#888", strokeDash=[4, 3])
                .encode(x="x:Q")
            )
            layers.append(parity)

        panel = alt.layer(*layers).properties(
            width=width,
            height=panel_height,
            title=alt.Title(
                text=f"{group_name} peers",
                anchor="start",
                fontSize=FONT_SIZE + 3,
                fontWeight="bold",
                color="#444",
            ),
        )
        panels.append(panel)

    return alt.concat(*panels, columns=2, spacing=14).properties(
        title=alt.Title(text=title, subtitle=subtitle)
    )


### Claude usage share

The following chart shows Algeria's share of all Claude conversations during the observation window, plotted against each peer group. Because this measure does not adjust for population, larger countries appear larger.

In [7]:
anthropic_peers = (
    df.loc[
        lambda d: (
            (d["geography"] == "country")
            & (d["facet"] == "country")
            & (d["variable"] == "usage_pct")
            & (d["geo_id"].isin(PEER_CODES))
        )
    ]
    .filter(["geo_id", "value"])
    .rename(columns={"value": "usage_pct"})
    .assign(
        country_name=lambda d: d["geo_id"].map(COUNTRY_NAMES),
        group=lambda d: d["geo_id"].map(GROUP_OF),
    )
)

anthropic_chart = make_peer_panel(
    anthropic_peers,
    value_col="usage_pct",
    title="Claude Usage Share for Algeria Against Each Peer Group",
    subtitle="% of global Claude conversations",
    value_axis_title="% of global Claude usage",
    value_format=".3f",
    width=300,
)
attaviz.add_caption(
    anthropic_chart,
    "Source: Anthropic Economic Index data",
)


alt.VConcatChart(...)

### Claude Usage Index (AUI)

The AUI expresses usage share by adjusting for population. A country's Claude usage share is divided by its share of the world's working-age population, where a value of 1.0 means proportionate use, values above 1.0 indicate more intensive use, values below indicate less.

The AUI is the more meaningful metric for understanding the intensity of AI adoption in a country, as it allows for comparison across countries of unequal size.

In [8]:
# Working-age population (15–64) for every AEI sample-eligible country.
wap = pd.read_csv(DATA_DIR / "AI" / "worldbank_working_age_population.csv")

SAMPLE_THRESHOLD = 250

# Non-country labels in the AEI raw data (drop them before querying WB).
NON_COUNTRY = {"NONE", "not_classified"}

eligible_codes = sorted(
    set(
        df.loc[
            (df["geography"] == "country")
            & (df["facet"] == "country")
            & (df["variable"] == "usage_count")
            & (df["value"] >= SAMPLE_THRESHOLD),
            "geo_id",
        ].tolist()
    )
    - NON_COUNTRY
)

In [9]:
# Anthropic Usage Index (AUI)

ELIGIBLE_USAGE_PCT_TOTAL = df.loc[
    (df["geography"] == "country")
    & (df["facet"] == "country")
    & (df["variable"] == "usage_pct")
    & (df["geo_id"].isin(eligible_codes)),
    "value",
].sum()
ELIGIBLE_WAP_TOTAL = wap["working_age_pop"].sum()

aui_peers = anthropic_peers.merge(
    wap.filter(["geo_id", "working_age_pop"]), on="geo_id", how="inner"
).assign(
    usage_share_eligible=lambda df: df["usage_pct"] / ELIGIBLE_USAGE_PCT_TOTAL * 100,
    wap_share_eligible=lambda df: df["working_age_pop"] / ELIGIBLE_WAP_TOTAL * 100,
    aui=lambda df: (
        (df["usage_pct"] / ELIGIBLE_USAGE_PCT_TOTAL)
        / (df["working_age_pop"] / ELIGIBLE_WAP_TOTAL)
    ),
    aui_label=lambda df: df["aui"].map(lambda v: f"{v:.2f}×"),
)

aui_chart = make_peer_panel(
    aui_peers,
    value_col="aui",
    title="Claude Usage Index (AUI) for Algeria Against Each Peer Group",
    subtitle="Claude usage share per working-age population share",
    value_axis_title="Usage Index (AUI)",
    value_format=".2f",
    parity_at=1.0,
    label_col="aui_label",
    width=300,
)
attaviz.add_caption(
    aui_chart,
    "Source: Anthropic Economic Index data",
)


alt.VConcatChart(...)

### ChatGPT country rank

OpenAI publishes a country ranking (not raw conversation counts) for ChatGPT in 2025. To make the ChatGPT view visually comparable to the Claude charts above, the rank is converted to a percentile within the 2025 country list, so higher bars mean more ChatGPT usage relative to other ranked countries. The metric is coarser than the Anthropic Economic Index and the observation window is annual rather than weekly, so this metric is used as a broad benchmark of AI adoption rather than a precise measure.

In [10]:
# OpenAI: country rank by share of ChatGPT messages (2025)
oai = pd.read_csv(
    DATA_DIR / "AI" / "OpenAI" / "share_of_messages_by_country_2025_rank.csv"
)
N_OPENAI_COUNTRIES = oai["country"].nunique()

openai_peers = (
    oai.loc[lambda df: df["country"].isin(PEER_CODES)]
    .rename(columns={"country": "geo_id"})
    .assign(
        country_name=lambda df: df["geo_id"].map(COUNTRY_NAMES),
        group=lambda df: df["geo_id"].map(GROUP_OF),
        # Convert rank to percentile so higher = more usage (visually consistent with the % charts above).
        adoption_percentile=lambda df: (
            (N_OPENAI_COUNTRIES - df["rank"]) / N_OPENAI_COUNTRIES * 100
        ),
    )
)

openai_chart = make_peer_panel(
    openai_peers,
    value_col="adoption_percentile",
    title="ChatGPT adoption for Algeria Against Each Peer Group",
    subtitle=f"Percentile of 2025 country ranking (higher = more usage) out of {N_OPENAI_COUNTRIES} countries",
    value_axis_title="Adoption percentile",
    value_format=".1f",
    width=300,
)
attaviz.add_caption(
    openai_chart,
    "Source: OpenAI Signals data",
)


alt.VConcatChart(...)

On raw Claude usage, Algeria sits in the middle among regional and structural peers, but at the bottom of its aspirational group. Algeria's 0.21% share of global Claude conversations places it fourth among regional peers and fourth among structural peers, but last when compared against Malaysia, Chile, Poland, and Romania. Because raw usage shares are not adjusted for population, this metric overstates the relative intensity of use in larger countries (Egypt, Vietnam) and understates it in smaller ones (Ghana, Jordan, Chile).

On the population-adjusted AUI, Algeria is the least intensive Claude user in every peer group. With an AUI of 0.33, Algeria ranks second to last among regional peers (Tunisia leads at 1.03, Morocco at 0.78), last among structural peers (Colombia 0.85, Peru 0.67), and last among aspirational peers, with Poland at 1.78 and Romania at 1.43, four to five times Algeria's intensity.

The ChatGPT picture is slightly better but tells the same broad story. Algeria sits at the 53rd percentile of OpenAI's 2025 ranking, which is third of six among regional peers (ahead of Morocco, Iraq, Egypt), fifth of six among structural peers (only Ghana behind), and last among aspirational peers. Given that Algeria's ChatGPT rank is relatively better than its AUI might suggest that the gap in Claude usage may reflect platform-specific factors rather than overall AI adoption. 

In [11]:
onet = pd.read_csv(
    ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/onet_task_statements.csv"
).assign(
    task_normalized=lambda df: df["Task"].str.lower().str.strip(),
    soc_str=lambda df: df["soc_major_group"].astype(int).astype(str).str.zfill(2),
)
task_to_soc = onet.filter(["task_normalized", "soc_str"]).drop_duplicates()

soc_struct = (
    pd.read_csv(
        ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/soc_structure.csv"
    )
    .dropna(subset=["Major Group"])
    .assign(
        soc_str=lambda df: df["soc_major_group"].astype(int).astype(str).str.zfill(2),
        soc_name=lambda df: df["SOC or O*NET-SOC 2019 Title"].str.replace(
            " Occupations", "", regex=False
        ),
    )
)
soc_name_map = dict(zip(soc_struct["soc_str"], soc_struct["soc_name"]))

tasks = dza.copy().query('facet == "onet_task" and variable == "onet_task_pct"')
real = (
    tasks.copy()
    .loc[lambda df: ~df["cluster_name"].isin(["none", "not_classified"])]
    .assign(task_normalized=lambda df: df["cluster_name"].str.lower().str.strip())
)

soc_df = (
    real.merge(task_to_soc, on="task_normalized", how="left")
    .groupby("soc_str", as_index=False)
    .agg(pct=("value", "sum"))
    .assign(soc_name=lambda df: df["soc_str"].map(soc_name_map))
    .sort_values("pct", ascending=False)
    .reset_index(drop=True)
    .drop(columns=["soc_str"])
    # add new row for "not classified" category to make sure the treemap adds up to 100%
    .pipe(
        lambda df: pd.concat(
            [
                df,
                pd.DataFrame(
                    {"soc_name": ["not_classified"], "pct": [100 - df["pct"].sum()]}
                ),
            ],
            ignore_index=True,
        )
    )
)

## What is Algeria using AI for?

The next question is what kind of work is being done with AI in Algeria? The Anthropic dataset offers two different classifications:

- By occupation (O\*NET): each conversation is matched to a U.S. Department of Labor task statement, then rolled up to a major occupational group. This classification connects directly to formal employment statistics, but conversations that don't correspond to any recognised occupation may fall outside the taxonomy.
- By user intent (request clusters): each conversation is placed in one of Anthropic's own clusters of what users are asking for. The classification is more inclusive of all conversations, as they are more focused on what users are doing with AI, rather than what their occupation is.

### By occupation

The following chart shows the occupational breakdown of Claude conversations in Algeria. The most common occupational groups dominate Algeria are Computer and Mathematical (coding, debugging, software design, statistical and data work) and Educational Instruction and Library (teaching, tutoring, and educational content). Everything else is below 3% individually. About 60% of conversations are labeled as "Other" and "Not classified", which means they don't correspond to a recognised occupation or task in the O\*NET taxonomy.

In [12]:
soc_treemap = make_treemap(
    soc_df,
    label_col="soc_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME} (Group by job)",
    subtitle="Categorized using O*NET-SOC codes",
)

attaviz.add_caption(
    soc_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [13]:
def request_df(level: int) -> pd.DataFrame:
    return (
        dza.loc[
            lambda df: (
                (df["facet"] == "request")
                & (df["variable"] == "request_pct")
                & (df["level"] == level)
            )
        ]
        .filter(["cluster_name", "value"])
        .copy()
        .rename(columns={"value": "pct"})
        .sort_values("pct", ascending=False)
        .reset_index(drop=True)
    )


req_l2 = request_df(2)
req_l1 = request_df(1)


### By user intent

The chart below shows the breakdown of conversations by user intent, according to Anthropic's own clustering. The most common cluster is assisting with writing, coding, and research work, which is consistent with the occupational classification. However, there are also some other clusters that are more specific to certain domains or tasks, such as assisting with math or science or providing general information.

In [14]:
l2_treemap = make_treemap(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L2)",
)

attaviz.add_caption(
    l2_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [15]:
l2_bar_chart = make_bar(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L2)",
    top_n=20,
)

attaviz.add_caption(
    l2_bar_chart,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

The two classifications both agree on the headline story: Software development (19% of conversations) and the combination of STEM homework (11%) and academic research (9%) account for the bulk of Algeria's Claude usage, and the same picture emerges in the occupational chart, where Computer and Mathematical and Educational Instruction and Library are the dominant categories. Coding and learning are what most Algerian Claude users are doing.

The request classification adds more nuance to this picture. Translation, writing, and editing tasks (10%) sit just below the headline and don't map cleanly to a single occupation, which translation crosses linguistic, legal, educational, and creative settings. Health and medical information (6%), creative content and marketing (6%), and daily-life questions (5%) are personal uses that occupational classifications largely miss.

In [16]:
l1_treemap = make_treemap(
    req_l1,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L1)",
    width=960,
)

attaviz.add_caption(
    l1_treemap,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

In [17]:
l1_bar_chart = make_bar(
    req_l1,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L1)",
    top_n=20,
)

attaviz.add_caption(
    l1_bar_chart,
    "Source: Anthropic Economic Index data",
)

alt.VConcatChart(...)

## What is distinctive about Algerian usage?

Setting aside global patterns that are common to most countries, what does Algeria do with Claude that the rest of the world is not? Two metrics help answer this question:

- Most frequent sorts request clusters by Algeria's share of conversations. It is a popularity ranking on what dominates Algeria's usage in absolute terms.
- Most distinctive sorts by the specialisation index, which is Algeria's share of a cluster divided by the global share of the same cluster. The "most distinctive" clusters are restricted to those reaching ≥1% in both Algeria and globally. Values above 1 mean Algeria over-uses that cluster relative to the world, while below 1 means under-use.

The five clusters that dominate Algeria's most-frequent list are academic coursework (6.5% of conversations), web development (5.5%), IT troubleshooting (3.7%), medical information (3.6%), and language learning and translation (3.5%). Together they account for roughly 23% of all classified conversations.

In [18]:
REQUEST_LEVEL = 1
NON_TASK_LABELS = {"not_classified", "none"}

_request_filter = (
    (df["facet"] == "request")
    & (df["variable"] == "request_pct")
    & (df["level"] == REQUEST_LEVEL)
)

dz_requests = (
    df.loc[_request_filter & (df["geo_id"] == COUNTRY_CODE), ["cluster_name", "value"]]
    .rename(columns={"value": "dz_pct"})
    .merge(
        df.loc[
            _request_filter & (df["geography"] == "global"), ["cluster_name", "value"]
        ].rename(columns={"value": "global_pct"}),
        on="cluster_name",
        how="inner",
    )
    .loc[lambda d: ~d["cluster_name"].isin(NON_TASK_LABELS)]
    .assign(specialization_index=lambda d: d["dz_pct"] / d["global_pct"])
    .reset_index(drop=True)
)

In [19]:
# Top 10 request clusters by share of Algeria conversations.
frequent_top = (
    dz_requests.sort_values("dz_pct", ascending=False)
    .head(10)
    .filter(["cluster_name", "dz_pct"])
    .rename(columns={"dz_pct": "pct"})
    .reset_index(drop=True)
)

frequent_chart = make_bar(
    frequent_top,
    label_col="cluster_name",
    value_col="pct",
    title=f"Most frequent requests in Algeria",
    subtitle="Share of classified Claude conversations",
    top_n=10,
    drop_not_classified=True,
)
attaviz.add_caption(
    frequent_chart,
    "Source: Anthropic Economic data",
)


alt.VConcatChart(...)

In terms of distinctiveness, the most over-represented cluster is language learning and translation help (2.5× the global rate), followed by translation-focused cluster (professional, academic, medical, and religious content) at 2.0×. Document and image extraction (2.1×), mathematics problems (1.8×), and medical information (1.8×) round out the top five distinctive uses.

Another interesting pattern emerges when comparing the most frequent and most distinctive clusters. Eight of the ten most distinctive clusters also appear among Algeria's ten most frequent, which means that the high-volume tasks are the distinctive ones.

In [20]:
# Top 10 request clusters where Algeria over-indexes vs. the world,
# restricted to clusters with at least 1% frequency in BOTH Algeria and globally.
NOISE_FLOOR_PCT = 1.0

distinctive_top = (
    dz_requests.loc[
        lambda d: (
            (d["dz_pct"] >= NOISE_FLOOR_PCT) & (d["global_pct"] >= NOISE_FLOOR_PCT)
        )
    ]
    .sort_values("specialization_index", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

row_h = 38
bar_size = 14
height_dist = max(220, row_h * len(distinctive_top))
x_max_dist = distinctive_top["specialization_index"].max() * 1.08
inside_cutoff_dist = x_max_dist * INSIDE_LABEL_THRESHOLD

base_dist = distinctive_top.assign(
    _zero=0.0,
    _index_label=lambda d: d["specialization_index"].map(lambda v: f"{v:.1f}×"),
)
sort_order = distinctive_top["cluster_name"].tolist()
y_enc_dist = alt.Y("cluster_name:N", sort=sort_order, title=None, axis=None)

bars_dist = (
    alt.Chart(base_dist)
    .mark_bar(size=bar_size)
    .encode(
        x=alt.X(
            "specialization_index:Q",
            title="Specialization index",
            scale=alt.Scale(domain=[0, x_max_dist], nice=False),
        ),
        y=y_enc_dist,
        tooltip=[
            alt.Tooltip("cluster_name:N", title="Request cluster"),
            alt.Tooltip("dz_pct:Q", title="% in Algeria", format=".2f"),
            alt.Tooltip("global_pct:Q", title="% globally", format=".2f"),
            alt.Tooltip("specialization_index:Q", title="Index", format=".2f"),
        ],
    )
)
labels_dist = (
    alt.Chart(base_dist)
    .mark_text(
        align="left",
        baseline="bottom",
        fontSize=FONT_SIZE,
        color="#222",
        dy=-(bar_size // 2) - 2,
    )
    .encode(x="_zero:Q", y=y_enc_dist, text="cluster_name:N")
)
# Long bars: white "1.5×" tucked inside the right edge of the bar.
index_text_inside = (
    alt.Chart(base_dist)
    .transform_filter(f"datum.specialization_index >= {inside_cutoff_dist}")
    .mark_text(
        align="right",
        baseline="middle",
        dx=-PAD,
        color="white",
        fontSize=FONT_SIZE - 1,
        fontWeight="bold",
    )
    .encode(x="specialization_index:Q", y=y_enc_dist, text="_index_label:N")
)
# Short bars: dark "1.5×" sitting just past the bar end.
index_text_outside = (
    alt.Chart(base_dist)
    .transform_filter(f"datum.specialization_index < {inside_cutoff_dist}")
    .mark_text(
        align="left",
        baseline="middle",
        dx=PAD,
        color="#444",
        fontSize=FONT_SIZE - 1,
    )
    .encode(x="specialization_index:Q", y=y_enc_dist, text="_index_label:N")
)
parity_rule = (
    alt.Chart(pd.DataFrame({"x": [1.0]}))
    .mark_rule(color="#888", strokeDash=[4, 3])
    .encode(x="x:Q")
)

distinctive_chart = (
    bars_dist + labels_dist + index_text_inside + index_text_outside + parity_rule
).properties(
    width=640,
    height=height_dist,
    title=alt.Title(
        text="Most distinctive requests in Algeria",
        subtitle=[
            "The classification is restricted to those with at least 1% share",
            "of conversations both in Algeria and globally",
        ],
    ),
)
attaviz.add_caption(distinctive_chart, "Source: Anthropic Economic data")


alt.VConcatChart(...)

Five limitations the reader should keep in mind:

- The Claude conversation snapshot covers 5–12 February 2026. Country rankings can shift across releases as platform reach evolves.
- Claude and ChatGPT together still understate the broader market, which includes Google Gemini, Meta AI, local Chinese models, and a growing universe of open-weight models. The analysis assumes Claude usage patterns are at least directionally representative of broader AI usage.